# RX Strategist OCR + Pipeline Demo

Run the full LangGraph workflow from a prescription **image**:

OCR → extract → RAG evidence → verify → knowledge graph → final checker

Add `GEMINI_API_KEY` to Colab Secrets before running. Use the bundled sample image, or upload a screenshot (`.png` / `.jpg` / `.webp`).

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/aravindpj/rx-strategist-mvp.git"

def find_src():
    for path in (Path("src"), Path("rx-strategist-mvp/src"), Path("../src")):
        if (path / "rx_strategist").is_dir():
            return path.resolve()
    return None

src = find_src()
if src is None:
    get_ipython().system(f"git clone {REPO_URL}")
    src = find_src()

if src is None:
    raise FileNotFoundError(
        "Could not find src/rx_strategist. "
        "Run this notebook from the repo folder, or clone "
        "https://github.com/aravindpj/rx-strategist-mvp.git"
    )

sys.path.insert(0, str(src))
req = src.parent / "requirements.txt"
get_ipython().run_line_magic("pip", f"install -q -r {req}")
print("Using", src)

In [ ]:
import sys
from pathlib import Path

for _src in (Path("src"), Path("rx-strategist-mvp/src"), Path("../src")):
    if (_src / "rx_strategist").is_dir():
        sys.path.insert(0, str(_src.resolve()))
        break
else:
    raise FileNotFoundError(
        "Could not find rx_strategist. Run the previous setup cell first."
    )

from google.colab import userdata
from rx_strategist.agents.workflow import run_workflow
from rx_strategist.evaluation.runner import evaluate

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets before running this notebook.")

SAMPLE_CANDIDATES = [
    Path("rx-strategist-mvp/data/prescriptions/sample_prescription.png"),
    Path("data/prescriptions/sample_prescription.png"),
    Path("../data/prescriptions/sample_prescription.png"),
]
IMAGE_PATH = next(path for path in SAMPLE_CANDIDATES if path.is_file())
IMAGE_PATH

In [ ]:
# Optional: upload a screenshot or photo instead of the sample image.
from google.colab import files

uploaded = files.upload()
if uploaded:
    filename, image_bytes = next(iter(uploaded.items()))
    suffix = Path(filename).suffix.lower()
    if suffix not in {".png", ".jpg", ".jpeg", ".webp"}:
        raise ValueError(f"Unsupported image type: {suffix}")
    IMAGE_PATH = Path(filename)
    IMAGE_PATH.write_bytes(image_bytes)
IMAGE_PATH

In [ ]:
result = run_workflow(image_path=str(IMAGE_PATH), api_key=api_key)
{
    "image": str(IMAGE_PATH),
    "ocr_text": result.get("ocr_text"),
    "verification": result["verification"]["overall_status"],
    "final_decision": result["final_check"]["final_decision"],
    "evidence": [hit["title"] for hit in result["evidence"]],
    "interactions": result["kg_context"]["interactions"],
}

In [ ]:
result["prescription"]

In [ ]:
result["evidence"]

In [ ]:
result["final_check"]

In [ ]:
report = evaluate()
report